In [1]:
import spacy
import re
import time
import warnings

import pandas as pd

from spacy.symbols import ORTH
from tqdm import tqdm
from pathlib import Path
from spacy.language import Language
from heuristic_tokenize import sent_tokenize_rules 

warnings.filterwarnings('ignore')

In [2]:
# Read MIMIC file and specify output directory
data_path = Path("../Data/discharge.csv.gz")

OUTPUT_DIR = Path("../Data/processed/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(data_path, usecols = ["subject_id","text"],nrows = 2000)

# Data cleaning steps
1. Split each note into sections using clinical-note rules
2. Use SciSpaCy to process each section
3. Add custom rules to detect sentence boundaries
4. Treat MIMIC de-identification placeholders as one token
5. Replace newlines inside sentences with spaces
6. Remove empty sentence strings
7. Write each sentence on its own line
8. Separate different notes with a blank line

In [3]:
#setting sentence boundaries
@Language.component("sbd_component")
def sbd_component(doc):
    for i, token in enumerate(doc[:-2]):
        # define sentence start if period + titlecase token
        if token.text == '.' and doc[i+1].is_title:
            doc[i+1].sent_start = True
        if token.text == '-' and doc[i+1].text != '-':
            doc[i+1].sent_start = True
    return doc

In [4]:
#convert de-identification text into one token
def fix_deid_tokens(text, processed_text):
    deid_regex  = r"\[\*\*.{0,15}.*?\*\*\]" 
    if text:
        indexes = [m.span() for m in re.finditer(deid_regex,text,flags=re.IGNORECASE)]
    else:
        indexes = []
    for start,end in indexes:
        processed_text.merge(start_idx=start,end_idx=end)
    return processed_text
    

In [5]:
def process_section(section, note, processed_sections):
    # perform spacy processing on section
    processed_section = nlp(section['sections'])
    processed_section = fix_deid_tokens(section['sections'], processed_section)
    processed_sections.append(processed_section)

In [6]:
def process_note_helper(note):
    note = re.sub(r'_{3,}', ' <PHI> ', note)
    note = re.sub(r'(\d+)-\s*\n\s*(\d+)', r'\1-\2', note)
    note_sections = sent_tokenize_rules(note)

    processed_sections = []
    for section, processed_section in zip(note_sections, nlp.pipe(note_sections)):
        processed_section = fix_deid_tokens(section, processed_section)
        processed_sections.append(processed_section)

    return processed_sections

In [7]:
def is_structured_block(text):
    """
    Detects sections that should keep their original line breaks,
    such as Past Medical History lists, labs, vitals, and medication lists.
    """
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    if len(lines) < 2:
        return False

    all_caps_lines = 0
    lab_lines = 0
    bullet_or_numbered_lines = 0
    colon_lines = 0

    for line in lines:
        clean_line = re.sub(r"\s+", " ", line).strip()

        # Example: ASTHMA/COPD, HYPERTENSION, ATRIAL FIBRILLATION
        if clean_line.isupper() and len(clean_line.split()) <= 8:
            all_caps_lines += 1

        # Example: WBC-7.2 RBC-4.06 Hgb-9.4
        if re.search(r"\b(WBC|RBC|Hgb|Hct|MCV|MCH|MCHC|RDW|Plt|Na|K|Cl|HCO3|Creat|Glucose|Calcium|Phos|Mg|PTT|INR)\b", clean_line):
            lab_lines += 1

        # Example: 1. Medication, - Hypertension
        if re.match(r"^(\d+\.|-|\*)", clean_line):
            bullet_or_numbered_lines += 1

        # Example: GENERAL:, HEENT:, CARDIAC:
        if re.match(r"^[A-Za-z /()]+:", clean_line):
            colon_lines += 1

    score = all_caps_lines + lab_lines + bullet_or_numbered_lines + colon_lines

    return score >= 2

In [8]:
def process_text(sent, note):
    sent_text = sent.text  # sent is now a spaCy Span, not a dict-like row

    if len(sent_text.strip()) == 0:
        return

    if "\n" in sent_text and is_structured_block(sent_text):
        lines = []
        for line in sent_text.splitlines():
            line = re.sub(r"\s+", " ", line).strip()
            if len(line) > 0:
                lines.append(line)
        note["text"] += "\n".join(lines) + "\n"
    else:
        sent_text = sent_text.replace("\n", " ")
        sent_text = re.sub(r"\s+", " ", sent_text).strip()
        note["text"] += sent_text + "\n"

In [9]:
def get_sentences(processed_section, note):
    for sent in processed_section.sents:
        process_text(sent, note)

In [10]:
def process_note(note):
    try:
        note_text = note["text"]
        if pd.isna(note_text):
            note["text"] = ""
            return note

        note_text = str(note_text)
        note["text"] = ""

        processed_sections = process_note_helper(note_text)
        for ps in processed_sections:
            get_sentences(ps, note)


        # Post-processing cleanup
        note["text"] = re.sub(r'(\d+)-\s*\n\s*(\d+)', r'\1-\2', note["text"])
        note["text"] = re.sub(r'\n\s*\.\s*\n', '.\n', note["text"])
        note["text"] = re.sub(r'\s+([.,;:?!])', r'\1', note["text"])
        note["text"] = re.sub(r'\n-\n', '\n- ', note["text"])
        note["text"] = re.sub(r'\s+-\s*\n', '\n- ', note["text"])
        note["text"] = re.sub(r'[ \t]+', ' ', note["text"])
        note["text"] = note["text"].strip()

        return note

    except Exception as e:
        print("Error processing note:", e)
        note["text"] = ""
        return note

In [11]:
start = time.time()
tqdm.pandas()

print('Begin reading notes')

print('Number of notes: %d' % len(data.index))
data['ind'] = list(range(len(data.index)))

nlp = spacy.load('en_core_sci_md', disable=['tagger', 'ner'])
nlp.tokenizer.add_special_case("<PHI>", [{ORTH: "<PHI>"}])
nlp.add_pipe("sbd_component", before='parser')

formatted_notes = data.progress_apply(process_note, axis=1)

# Rename cleaned text column
cleaned_df = formatted_notes.rename(columns={"text": "cleaned_text"})

# Keep subject_id attached to each cleaned note
cleaned_df = cleaned_df[["subject_id", "cleaned_text"]].copy()

# Remove failed/empty notes
cleaned_df = cleaned_df.dropna(subset=["cleaned_text"])
cleaned_df = cleaned_df[cleaned_df["cleaned_text"].str.strip() != ""]

# Save one CSV file
output_file = OUTPUT_DIR / "discharge_cleaned.csv"

cleaned_df.to_csv(output_file, index=False)

end = time.time()
print(end - start)
print("Done formatting notes")
print("Saved to:", output_file)

Begin reading notes
Number of notes: 2000


100%|██████████| 2000/2000 [11:10<00:00,  2.98it/s]


675.6210687160492
Done formatting notes
Saved to: ../Data/processed/discharge_cleaned.csv


In [12]:
idx = 2  # choose any row number you want to inspect

print("ORIGINAL NOTE")
print("=" * 80)
print(data.loc[idx, "text"])

print("\n\nCLEANED NOTE")
print("=" * 80)
print(cleaned_df.loc[idx, "cleaned_text"])

ORIGINAL NOTE
 
Name:  ___                     Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: MEDICINE
 
Allergies: 
Percocet / Vicodin
 
Attending: ___
 
Chief Complaint:
altered mental status
 
Major Surgical or Invasive Procedure:
none
 
History of Present Illness:
Mrs. ___ is a ___ female with HIV on HAART, COPD, HCV 
cirrhosis complicated by ascites and hepatic encephalopathy who 
initially presented to the ED yesterday with hypotension after a 
paracentesis.  
The patient has had accelerated decompensation of her cirrhosis 
recently with worsening ascites, and she is maintained on twice 
weekly paracentesis. She was at her regular session yesterday 
when she had hypotension to SBP ___ and felt lightheadedness. 
Per the patient, that's when her memory started to get fuzzy. 
She does not have much recollection of what happened since then. 
Her outpatient hepatologist saw her and recommended that she go 